In [1]:
from pathlib import Path
import hashlib
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


# Optional outputs
OUTPUT_DIR = Path("./data/interim")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_OUTPUT = OUTPUT_DIR / "canonical_player_fixtures.parquet"
AUDIT_OUTPUT = OUTPUT_DIR / "data_audit_summary.json"

In [2]:
path_downloaded = kagglehub.dataset_download("joebeachcapital/fantasy-football")

Using Colab cache for faster access to the 'fantasy-football' dataset.


In [3]:
path_global = os.path.join(path_downloaded, "cleaned_merged_seasons.csv")

## 1. Load data without mutating it
#### The raw source is kept immutable in memory. All normalization occurs on a copy.
#### This prevents exploratory cleanup from changing the evidence we are auditing.

In [4]:
raw = pd.read_csv(path_global)
df = raw.copy()

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

Rows: 96169
Columns: 37


/tmp/ipykernel_34718/3672073749.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(path_global)


,season_x,name,position,team_x,assists,bonus,bps,clean_sheets,creativity,element,fixture,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,opponent_team,opp_team_name,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,2016-17,Aaron Cresswell,DEF,NaN,0,0,0,0,0.0,454,10,0,0,0.0,0.0,2016-08-15T19:00:00Z,0,4,Chelsea,0,0,0,0,1,0,14023,1.0,2.0,0.0,0,0,0,0,55,False,0,1
1,2016-17,Aaron Lennon,MID,NaN,0,0,6,0,0.3,142,3,0,0,0.9,8.2,2016-08-13T14:00:00Z,15,17,Spurs,0,0,0,0,1,0,13918,1.0,1.0,0.0,1,0,0,0,60,True,0,1
2,2016-17,Aaron Ramsey,MID,NaN,0,0,5,0,4.9,16,8,3,0,3.0,2.2,2016-08-14T15:00:00Z,60,9,Liverpool,0,0,0,0,1,0,163170,4.0,3.0,23.0,2,0,0,0,80,True,0,1
3,2016-17,Abdoulaye Doucouré,MID,NaN,0,0,0,0,0.0,482,7,0,0,0.0,0.0,2016-08-13T14:00:00Z,0,13,Southampton,0,0,0,0,1,0,1051,1.0,1.0,0.0,0,0,0,0,50,False,0,1
4,2016-17,Adam Forshaw,MID,NaN,0,0,3,0,1.3,286,6,1,0,0.3,2.0,2016-08-13T14:00:00Z,69,14,Stoke,0,0,0,0,1,0,2723,1.0,1.0,0.0,1,0,0,0,45,True,1,1


In [5]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_n": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

display(schema)

,column,dtype,missing_n,missing_pct,n_unique
3,team_x,object,19852,20.642827,25
36,GW,int64,0,0.000000,38
4,assists,int64,0,0.000000,5
5,bonus,int64,0,0.000000,4
6,bps,int64,0,0.000000,114
7,clean_sheets,int64,0,0.000000,2
8,creativity,float64,0,0.000000,830
9,element,int64,0,0.000000,778
10,fixture,int64,0,0.000000,380
11,goals_conceded,int64,0,0.000000,10


## 2. Normalize naming only 
We create canonical aliases while preserving the original columns until the audit is complete.
### The source notebook uses names such as:
- season_x
- element
- name
- GW
- team_x
- opp_team_name
### Canonical names are easier to reason about:
- season
- player_id
- player_name
- gameweek
- team
- opponent_team

In [6]:
ALIASES = {
    "season_x": "season",
    "element": "player_id",
    "name": "player_name",
    "GW": "gameweek",
    "team_x": "team",
    "opp_team_name": "opponent_team",
}

rename_map = {old: new for old, new in ALIASES.items() if old in df.columns and new not in df.columns}
df = df.rename(columns=rename_map)

if "kickoff_time" in df.columns:
    df["kickoff_time"] = pd.to_datetime(df["kickoff_time"], errors="coerce", utc=True)

if "gameweek" in df.columns:
    df["gameweek"] = pd.to_numeric(df["gameweek"], errors="coerce").astype("Int64")

if "player_id" in df.columns:
    df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")

display(df.head())

,season,player_name,position,team,assists,bonus,bps,clean_sheets,creativity,player_id,fixture,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,opponent_team,opp_team_name,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,gameweek
0,2016-17,Aaron Cresswell,DEF,NaN,0,0,0,0,0.0,454,10,0,0,0.0,0.0,2016-08-15 19:00:00+00:00,0,4,Chelsea,0,0,0,0,1,0,14023,1.0,2.0,0.0,0,0,0,0,55,False,0,1
1,2016-17,Aaron Lennon,MID,NaN,0,0,6,0,0.3,142,3,0,0,0.9,8.2,2016-08-13 14:00:00+00:00,15,17,Spurs,0,0,0,0,1,0,13918,1.0,1.0,0.0,1,0,0,0,60,True,0,1
2,2016-17,Aaron Ramsey,MID,NaN,0,0,5,0,4.9,16,8,3,0,3.0,2.2,2016-08-14 15:00:00+00:00,60,9,Liverpool,0,0,0,0,1,0,163170,4.0,3.0,23.0,2,0,0,0,80,True,0,1
3,2016-17,Abdoulaye Doucouré,MID,NaN,0,0,0,0,0.0,482,7,0,0,0.0,0.0,2016-08-13 14:00:00+00:00,0,13,Southampton,0,0,0,0,1,0,1051,1.0,1.0,0.0,0,0,0,0,50,False,0,1
4,2016-17,Adam Forshaw,MID,NaN,0,0,3,0,1.3,286,6,1,0,0.3,2.0,2016-08-13 14:00:00+00:00,69,14,Stoke,0,0,0,0,1,0,2723,1.0,1.0,0.0,1,0,0,0,45,True,1,1


### 3. Establish row identity
A production-quality feature pipeline needs a stable fixture identifier.\
Preferred key:  
season, fixture_id, player_id  
Fallback diagnostic key:  
season, kickoff_time, team, opponent_team, player_id\
\
The fallback is not as strong as a provider-issued fixture ID, so if the raw source lacks fixture_id, sourcing one should become a backlog item before productionization.

In [7]:
preferred_key = [c for c in ["season", "fixture_id", "player_id"] if c in df.columns]

if set(["season", "fixture_id", "player_id"]).issubset(df.columns):
    row_key = ["season", "fixture_id", "player_id"]
    key_quality = "preferred"
else:
    fallback = ["season", "kickoff_time", "team", "opponent_team", "player_id"]
    row_key = [c for c in fallback if c in df.columns]
    key_quality = "fallback"

print("Key quality:", key_quality)
print("Row key:", row_key)

duplicate_mask = df.duplicated(row_key, keep=False) if row_key else pd.Series(False, index=df.index)
print("Duplicate rows under chosen key:", int(duplicate_mask.sum()))

if duplicate_mask.any():
    display(df.loc[duplicate_mask].sort_values(row_key).head(100))

Key quality: fallback
Row key: ['season', 'kickoff_time', 'team', 'opponent_team', 'player_id']
Duplicate rows under chosen key: 0


In [8]:
def assert_unique(data, keys, label):
    duplicated = data.duplicated(keys, keep=False)
    if duplicated.any():
        raise AssertionError(
            f"{label}: {duplicated.sum()} rows violate uniqueness for {keys}"
        )


# 4. Audit missingness — no blanket deletion
The reference notebook eventually calls dropna() over the whole engineered table.\
We replace that with a column-specific policy.\
Suggested categories\

**Hard-required identity fields**  
- season
- player_id
- kickoff_time
- team
- opponent_team
- position
- gameweek\

**Outcome/stat fields**
- minutes
- total_points
- goals_scored
- assists
- etc.
Missing outcome/stat fields require investigation before zero-filling because NaN can mean either:
1. true zero not recorded;
2. unavailable source field;
3. malformed join.
Optional/context fields
May remain nullable if their meaning is documented.

In [9]:
required_identity = [
    c for c in
    ["season", "player_id", "kickoff_time", "team", "opponent_team", "position", "gameweek"]
    if c in df.columns
]

missing_required = pd.DataFrame({
    "column": required_identity,
    "missing_n": [int(df[c].isna().sum()) for c in required_identity],
    "missing_pct": [float(df[c].isna().mean() * 100) for c in required_identity],
})

display(missing_required)

,column,missing_n,missing_pct
0,season,0,0.000000
1,player_id,0,0.000000
2,kickoff_time,0,0.000000
3,team,19852,20.642827
4,opponent_team,0,0.000000
5,position,0,0.000000
6,gameweek,0,0.000000


In [10]:
# Missing team coverage by season and gameweek
if {"season", "gameweek", "team", "player_id"}.issubset(df.columns):
    team_missing_by_period = (
        df.assign(team_missing=df["team"].isna())
        .groupby(["season", "gameweek"], as_index=False)
        .agg(
            rows=("player_id", "size"),
            missing_team_n=("team_missing", "sum"),
            missing_team_pct=("team_missing", lambda s: s.mean() * 100),
        )
    )

    display(team_missing_by_period)

    print("\nSeason-level summary:")
    team_missing_by_season = (
        df.groupby("season", as_index=False)
        .agg(
            rows=("player_id", "size"),
            missing_team_n=("team", lambda s: s.isna().sum()),
            observed_team_n=("team", lambda s: s.notna().sum()),
        )
    )

    team_missing_by_season["missing_team_pct"] = (
        team_missing_by_season["missing_team_n"]
        / team_missing_by_season["rows"]
        * 100
    )

    display(team_missing_by_season)

,season,gameweek,rows,missing_team_n,missing_team_pct
0,2016-17,1,195,195,100.0
1,2016-17,2,198,198,100.0
2,2016-17,3,201,201,100.0
3,2016-17,4,213,213,100.0
4,2016-17,5,213,213,100.0
5,2016-17,6,213,213,100.0
6,2016-17,7,213,213,100.0
7,2016-17,8,214,214,100.0
8,2016-17,9,216,216,100.0
9,2016-17,10,218,218,100.0



Season-level summary:


,season,rows,missing_team_n,observed_team_n,missing_team_pct
0,2016-17,8567,8567,0,100.0
1,2017-18,11285,11285,0,100.0
2,2020-21,24365,0,24365,0.0
3,2021-22,25447,0,25447,0.0
4,2022-23,26505,0,26505,0.0


In [11]:
fixture_candidates = [
    "fixture",
    "fixture_id",
    "team",
    "team_name",
    "opponent_team",
    "opp_team_name",
    "was_home",
    "kickoff_time",
    "gameweek",
    "season",
]

available_fixture_cols = [
    c for c in fixture_candidates
    if c in df.columns
]

print("Available fixture-related columns:")
display(pd.DataFrame({"column": available_fixture_cols}))

if available_fixture_cols:
    fixture_missingness = pd.DataFrame({
        "column": available_fixture_cols,
        "missing_n": [
            int(df[c].isna().sum())
            for c in available_fixture_cols
        ],
        "missing_pct": [
            float(df[c].isna().mean() * 100)
            for c in available_fixture_cols
        ],
        "n_unique": [
            int(df[c].nunique(dropna=True))
            for c in available_fixture_cols
        ],
    })

    display(fixture_missingness)

Available fixture-related columns:


,column
0,fixture
1,team
2,opponent_team
3,opp_team_name
4,was_home
5,kickoff_time
6,gameweek
7,season


,column,missing_n,missing_pct,n_unique
0,fixture,0,0.000000,380
1,team,19852,20.642827,25
2,opponent_team,0,0.000000,20
3,opp_team_name,0,0.000000,31
4,was_home,0,0.000000,2
5,kickoff_time,0,0.000000,1220
6,gameweek,0,0.000000,38
7,season,0,0.000000,5


In [12]:
if "team" in df.columns:
    cols = [
        c for c in [
            "season",
            "gameweek",
            "fixture",
            "fixture_id",
            "kickoff_time",
            "player_id",
            "player_name",
            "team",
            "team_name",
            "opponent_team",
            "opp_team_name",
            "was_home",
        ]
        if c in df.columns
    ]

    display(
        df.loc[df["team"].isna(), cols]
        .head(100)
    )

,season,gameweek,fixture,kickoff_time,player_id,player_name,team,opponent_team,opp_team_name,was_home
0,2016-17,1,10,2016-08-15 19:00:00+00:00,454,Aaron Cresswell,NaN,4,Chelsea,False
1,2016-17,1,3,2016-08-13 14:00:00+00:00,142,Aaron Lennon,NaN,17,Spurs,True
2,2016-17,1,8,2016-08-14 15:00:00+00:00,16,Aaron Ramsey,NaN,9,Liverpool,True
3,2016-17,1,7,2016-08-13 14:00:00+00:00,482,Abdoulaye Doucouré,NaN,13,Southampton,False
4,2016-17,1,6,2016-08-13 14:00:00+00:00,286,Adam Forshaw,NaN,14,Stoke,True
5,2016-17,1,8,2016-08-14 15:00:00+00:00,205,Adam Lallana,NaN,1,Arsenal,False
6,2016-17,1,9,2016-08-14 12:30:00+00:00,34,Adam Smith,NaN,11,Man Utd,True
7,2016-17,1,10,2016-08-15 19:00:00+00:00,450,Adrián San Miguel del Castillo,NaN,4,Chelsea,False
8,2016-17,1,8,2016-08-14 15:00:00+00:00,21,Alex Iwobi,NaN,9,Liverpool,True
9,2016-17,1,7,2016-08-13 14:00:00+00:00,101,Alex McCarthy,NaN,18,Watford,True


# 5. Investigate missing team values explicitly
The reference notebook's populate_team_x() computes a corrected dataframe and returns the original dataframe, so the fix is ineffective.\
Here, no team value is filled unless we can demonstrate that the inferred value is unambiguous.\
A safe recovery rule may use known fixture/opponent relationships, but every repaired row must be counted and auditable.

In [13]:
# ============================================================
# CELL 17 — AUDIT FIXTURE STRUCTURE FOR TEAM RECONSTRUCTION
# ============================================================

required_cols = {
    "season",
    "fixture",
    "opponent_team",
    "team",
    "was_home",
}

missing_required = required_cols - set(df.columns)

if missing_required:
    raise KeyError(
        f"Missing required columns for fixture-based team audit: "
        f"{sorted(missing_required)}"
    )


fixture_team_structure = (
    df.groupby(["season", "fixture"], as_index=False)
    .agg(
        rows=("player_id", "size"),
        unique_opponents=(
            "opponent_team",
            lambda s: tuple(sorted(s.dropna().unique()))
        ),
        n_unique_opponents=(
            "opponent_team",
            lambda s: s.dropna().nunique()
        ),
        missing_opponent_n=(
            "opponent_team",
            lambda s: s.isna().sum()
        ),
        home_flags=(
            "was_home",
            lambda s: tuple(sorted(s.dropna().unique()))
        ),
        n_home_flags=(
            "was_home",
            lambda s: s.dropna().nunique()
        ),
    )
)

display(
    fixture_team_structure[
        [
            "season",
            "fixture",
            "rows",
            "unique_opponents",
            "n_unique_opponents",
            "missing_opponent_n",
            "home_flags",
            "n_home_flags",
        ]
    ].head(20)
)


print("\nNumber of unique opponent teams per fixture:")
display(
    fixture_team_structure[
        "n_unique_opponents"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_unique_opponents")
    .reset_index(name="fixtures")
)


print("\nMissing opponent_team values:")
print(
    fixture_team_structure[
        "missing_opponent_n"
    ].sum()
)


print("\nFixtures that do NOT resolve to exactly two teams:")
bad_fixture_structure = fixture_team_structure[
    fixture_team_structure["n_unique_opponents"] != 2
]

display(bad_fixture_structure)

,season,fixture,rows,unique_opponents,n_unique_opponents,missing_opponent_n,home_flags,n_home_flags
0,2016-17,1,14,"(3, 16)",2,0,"(False, True)",2
1,2016-17,2,16,"(5, 19)",2,0,"(False, True)",2
2,2016-17,3,26,"(6, 17)",2,0,"(False, True)",2
3,2016-17,4,17,"(7, 8)",2,0,"(False, True)",2
4,2016-17,5,14,"(10, 15)",2,0,"(False, True)",2
5,2016-17,6,6,"(12, 14)",2,0,"(False, True)",2
6,2016-17,7,22,"(13, 18)",2,0,"(False, True)",2
7,2016-17,8,29,"(1, 9)",2,0,"(False, True)",2
8,2016-17,9,26,"(2, 11)",2,0,"(False, True)",2
9,2016-17,10,25,"(4, 20)",2,0,"(False, True)",2



Number of unique opponent teams per fixture:


,n_unique_opponents,fixtures
0,2,1900



Missing opponent_team values:
0

Fixtures that do NOT resolve to exactly two teams:


,season,fixture,rows,unique_opponents,n_unique_opponents,missing_opponent_n,home_flags,n_home_flags


In [14]:
# ============================================================
# CELL 18 — DERIVE TEAM FROM FIXTURE OPPONENTS
#            (AUDIT ONLY — DOES NOT MODIFY df)
# ============================================================

fixture_team_lookup = {
    (row.season, row.fixture): set(row.unique_opponents)
    for row in fixture_team_structure.itertuples()
    if row.n_unique_opponents == 2
}


def infer_team_from_fixture(row):
    """
    Infer a player's team from the two teams represented by
    opponent_team within the same fixture.

    If fixture teams are {A, B} and opponent_team is B,
    then the player's team must be A.
    """
    fixture_teams = fixture_team_lookup.get(
        (row["season"], row["fixture"])
    )

    if fixture_teams is None:
        return pd.NA

    opponent = row["opponent_team"]

    if pd.isna(opponent):
        return pd.NA

    candidates = fixture_teams - {opponent}

    if len(candidates) != 1:
        return pd.NA

    return next(iter(candidates))


team_recovery_audit = df[
    [
        "season",
        "fixture",
        "gameweek",
        "player_id",
        "player_name",
        "team",
        "opponent_team",
        "opp_team_name",
        "was_home",
    ]
].copy()

team_recovery_audit["fixture_inferred_team"] = (
    df.apply(infer_team_from_fixture, axis=1)
)

team_recovery_audit["recovery_status"] = "not_needed"

missing_mask = team_recovery_audit["team"].isna()

team_recovery_audit.loc[
    missing_mask
    & team_recovery_audit["fixture_inferred_team"].notna(),
    "recovery_status",
] = "recoverable"

team_recovery_audit.loc[
    missing_mask
    & team_recovery_audit["fixture_inferred_team"].isna(),
    "recovery_status",
] = "unresolved"


print("Team recovery audit:")
display(
    team_recovery_audit["recovery_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="rows")
)


print("\nRecovery by season:")
display(
    team_recovery_audit.groupby(
        ["season", "recovery_status"],
        as_index=False,
    ).size()
)


print("\nExample reconstructed rows:")
display(
    team_recovery_audit.loc[
        team_recovery_audit["recovery_status"] == "recoverable"
    ].head(30)
)


print("\nUnresolved rows:")
display(
    team_recovery_audit.loc[
        team_recovery_audit["recovery_status"] == "unresolved"
    ].head(50)
)

Team recovery audit:


,status,rows
0,not_needed,76317
1,recoverable,19852



Recovery by season:


,season,recovery_status,size
0,2016-17,recoverable,8567
1,2017-18,recoverable,11285
2,2020-21,not_needed,24365
3,2021-22,not_needed,25447
4,2022-23,not_needed,26505



Example reconstructed rows:


,season,fixture,gameweek,player_id,player_name,team,opponent_team,opp_team_name,was_home,fixture_inferred_team,recovery_status
0,2016-17,10,1,454,Aaron Cresswell,NaN,4,Chelsea,False,20,recoverable
1,2016-17,3,1,142,Aaron Lennon,NaN,17,Spurs,True,6,recoverable
2,2016-17,8,1,16,Aaron Ramsey,NaN,9,Liverpool,True,1,recoverable
3,2016-17,7,1,482,Abdoulaye Doucouré,NaN,13,Southampton,False,18,recoverable
4,2016-17,6,1,286,Adam Forshaw,NaN,14,Stoke,True,12,recoverable
5,2016-17,8,1,205,Adam Lallana,NaN,1,Arsenal,False,9,recoverable
6,2016-17,9,1,34,Adam Smith,NaN,11,Man Utd,True,2,recoverable
7,2016-17,10,1,450,Adrián San Miguel del Castillo,NaN,4,Chelsea,False,20,recoverable
8,2016-17,8,1,21,Alex Iwobi,NaN,9,Liverpool,True,1,recoverable
9,2016-17,7,1,101,Alex McCarthy,NaN,18,Watford,True,13,recoverable



Unresolved rows:


,season,fixture,gameweek,player_id,player_name,team,opponent_team,opp_team_name,was_home,fixture_inferred_team,recovery_status


In [15]:
# ============================================================
# CELL 19A — BUILD AND VALIDATE TEAM ID -> TEAM NAME MAPPING
# ============================================================

team_id_name_pairs = (
    df[
        ["season", "opponent_team", "opp_team_name"]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values(
        ["season", "opponent_team", "opp_team_name"]
    )
)

display(team_id_name_pairs.head(50))


# Check whether an opponent_team ID maps to more than one
# team name inside the same season.
mapping_conflicts = (
    team_id_name_pairs
    .groupby(
        ["season", "opponent_team"]
    )["opp_team_name"]
    .nunique()
    .reset_index(name="n_names")
)

mapping_conflicts = mapping_conflicts[
    mapping_conflicts["n_names"] != 1
]

print(
    "Season/team-ID mappings with multiple names:",
    len(mapping_conflicts),
)

display(mapping_conflicts)

,season,opponent_team,opp_team_name
5,2016-17,1,Arsenal
11,2016-17,2,Bournemouth
61,2016-17,3,Burnley
0,2016-17,4,Chelsea
25,2016-17,5,Crystal Palace
22,2016-17,6,Everton
23,2016-17,7,Hull
12,2016-17,8,Leicester
2,2016-17,9,Liverpool
103,2016-17,10,Man City


Season/team-ID mappings with multiple names: 0


,season,opponent_team,n_names


In [16]:
# ============================================================
# CELL 19B — CONVERT INFERRED TEAM IDS TO TEAM NAMES
# ============================================================

team_name_lookup = (
    team_id_name_pairs
    .drop_duplicates(
        ["season", "opponent_team"]
    )
    .set_index(
        ["season", "opponent_team"]
    )["opp_team_name"]
    .to_dict()
)


def inferred_team_name(row):
    team_id = row["fixture_inferred_team"]

    if pd.isna(team_id):
        return pd.NA

    return team_name_lookup.get(
        (row["season"], team_id),
        pd.NA,
    )


team_recovery_audit[
    "fixture_inferred_team_name"
] = team_recovery_audit.apply(
    inferred_team_name,
    axis=1,
)


display(
    team_recovery_audit[
        [
            "season",
            "fixture",
            "player_name",
            "team",
            "opponent_team",
            "opp_team_name",
            "fixture_inferred_team",
            "fixture_inferred_team_name",
        ]
    ].head(30)
)

,season,fixture,player_name,team,opponent_team,opp_team_name,fixture_inferred_team,fixture_inferred_team_name
0,2016-17,10,Aaron Cresswell,NaN,4,Chelsea,20,West Ham
1,2016-17,3,Aaron Lennon,NaN,17,Spurs,6,Everton
2,2016-17,8,Aaron Ramsey,NaN,9,Liverpool,1,Arsenal
3,2016-17,7,Abdoulaye Doucouré,NaN,13,Southampton,18,Watford
4,2016-17,6,Adam Forshaw,NaN,14,Stoke,12,Middlesbrough
5,2016-17,8,Adam Lallana,NaN,1,Arsenal,9,Liverpool
6,2016-17,9,Adam Smith,NaN,11,Man Utd,2,Bournemouth
7,2016-17,10,Adrián San Miguel del Castillo,NaN,4,Chelsea,20,West Ham
8,2016-17,8,Alex Iwobi,NaN,9,Liverpool,1,Arsenal
9,2016-17,7,Alex McCarthy,NaN,18,Watford,13,Southampton


In [17]:
# ============================================================
# CELL 19C — VALIDATE FIXTURE INFERENCE AGAINST KNOWN TEAM NAME
# ============================================================

known_team_validation = team_recovery_audit[
    team_recovery_audit["team"].notna()
    & team_recovery_audit[
        "fixture_inferred_team_name"
    ].notna()
].copy()


known_team_validation["matches_observed"] = (
    known_team_validation["team"]
    .astype(str)
    .str.strip()
    .str.casefold()
    ==
    known_team_validation[
        "fixture_inferred_team_name"
    ]
    .astype(str)
    .str.strip()
    .str.casefold()
)


validation_summary = (
    known_team_validation
    .groupby("season", as_index=False)
    .agg(
        rows=("player_id", "size"),
        matches=("matches_observed", "sum"),
        mismatches=(
            "matches_observed",
            lambda s: (~s).sum()
        ),
        accuracy=(
            "matches_observed",
            "mean"
        ),
    )
)

validation_summary["accuracy_pct"] = (
    validation_summary["accuracy"] * 100
)

display(validation_summary)


overall_accuracy = (
    known_team_validation[
        "matches_observed"
    ].mean()
)

print(
    f"Overall fixture reconstruction accuracy: "
    f"{overall_accuracy:.6%}"
)


mismatches = known_team_validation[
    ~known_team_validation["matches_observed"]
]

print(
    f"Observed-vs-inferred mismatches: "
    f"{len(mismatches):,}"
)

display(
    mismatches[
        [
            "season",
            "fixture",
            "player_name",
            "team",
            "fixture_inferred_team",
            "fixture_inferred_team_name",
            "opponent_team",
            "opp_team_name",
        ]
    ].head(50)
)

,season,rows,matches,mismatches,accuracy,accuracy_pct
0,2020-21,24365,24365,0,1.0,100.0
1,2021-22,25447,25447,0,1.0,100.0
2,2022-23,26505,26505,0,1.0,100.0


Overall fixture reconstruction accuracy: 100.000000%
Observed-vs-inferred mismatches: 0


,season,fixture,player_name,team,fixture_inferred_team,fixture_inferred_team_name,opponent_team,opp_team_name


## Consolidated Repair Cell

In [18]:
# ============================================================
# CANONICAL TEAM RECONSTRUCTION
# ============================================================

df_clean = df.copy()

# ------------------------------------------------------------
# 1. Build season-specific team ID -> name mapping
# ------------------------------------------------------------

team_id_name_map = (
    df_clean[
        ["season", "opponent_team", "opp_team_name"]
    ]
    .dropna()
    .drop_duplicates()
)

mapping_conflicts = (
    team_id_name_map
    .groupby(["season", "opponent_team"])["opp_team_name"]
    .nunique()
)

assert (mapping_conflicts == 1).all(), (
    "Some season/team IDs map to multiple team names."
)

team_name_lookup = (
    team_id_name_map
    .drop_duplicates(["season", "opponent_team"])
    .set_index(["season", "opponent_team"])["opp_team_name"]
    .to_dict()
)

# ------------------------------------------------------------
# 2. Build fixture -> two participating team IDs
# ------------------------------------------------------------

fixture_team_ids = (
    df_clean
    .groupby(["season", "fixture"])["opponent_team"]
    .agg(lambda s: tuple(sorted(s.dropna().unique())))
)

invalid_fixtures = fixture_team_ids[
    fixture_team_ids.apply(len) != 2
]

assert invalid_fixtures.empty, (
    f"{len(invalid_fixtures)} fixtures do not resolve "
    "to exactly two teams."
)

fixture_team_lookup = {
    key: set(team_ids)
    for key, team_ids in fixture_team_ids.items()
}

# ------------------------------------------------------------
# 3. Infer team ID from fixture + opponent
# ------------------------------------------------------------

def infer_team_id(row):
    fixture_teams = fixture_team_lookup.get(
        (row["season"], row["fixture"])
    )

    if fixture_teams is None:
        return pd.NA

    opponent = row["opponent_team"]

    if pd.isna(opponent):
        return pd.NA

    candidates = fixture_teams - {opponent}

    if len(candidates) != 1:
        return pd.NA

    return next(iter(candidates))


df_clean["_inferred_team_id"] = df_clean.apply(
    infer_team_id,
    axis=1,
)

# ------------------------------------------------------------
# 4. Convert inferred team ID to season-specific team name
# ------------------------------------------------------------

def infer_team_name(row):
    team_id = row["_inferred_team_id"]

    if pd.isna(team_id):
        return pd.NA

    return team_name_lookup.get(
        (row["season"], team_id),
        pd.NA,
    )


df_clean["_inferred_team_name"] = df_clean.apply(
    infer_team_name,
    axis=1,
)

# ------------------------------------------------------------
# 5. Validate inference against rows with known team
# ------------------------------------------------------------

known_mask = (
    df_clean["team"].notna()
    & df_clean["_inferred_team_name"].notna()
)

known_match = (
    df_clean.loc[known_mask, "team"]
    .astype(str)
    .str.strip()
    .str.casefold()
    ==
    df_clean.loc[known_mask, "_inferred_team_name"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

assert known_match.all(), (
    f"{(~known_match).sum()} known team rows disagree "
    "with fixture-based reconstruction."
)

# ------------------------------------------------------------
# 6. Repair missing teams only
# ------------------------------------------------------------

missing_before = df_clean["team"].isna()

repairable = (
    missing_before
    & df_clean["_inferred_team_name"].notna()
)

df_clean.loc[
    repairable,
    "team",
] = df_clean.loc[
    repairable,
    "_inferred_team_name",
]

# ------------------------------------------------------------
# 7. Validate final result
# ------------------------------------------------------------

assert repairable.sum() == missing_before.sum(), (
    "Not all missing team values were recoverable."
)

assert df_clean["team"].isna().sum() == 0, (
    "Missing team values remain after reconstruction."
)

assert not (
    df_clean["team"]
    .astype(str)
    .str.strip()
    .str.casefold()
    ==
    df_clean["opp_team_name"]
    .astype(str)
    .str.strip()
    .str.casefold()
).any(), (
    "Found rows where team equals opponent."
)

# ------------------------------------------------------------
# 8. Record provenance
# ------------------------------------------------------------

df_clean["team_recovery_method"] = "observed"

df_clean.loc[
    missing_before,
    "team_recovery_method",
] = "fixture_opponent_inference"

print(f"Original missing team rows: {missing_before.sum():,}")
print(f"Rows reconstructed:         {repairable.sum():,}")
print(f"Missing team after repair:  {df_clean['team'].isna().sum():,}")

display(
    df_clean["team_recovery_method"]
    .value_counts()
    .rename_axis("method")
    .reset_index(name="rows")
)

# Remove temporary columns
df_clean = df_clean.drop(
    columns=[
        "_inferred_team_id",
        "_inferred_team_name",
    ]
)

Original missing team rows: 19,852
Rows reconstructed:         19,852
Missing team after repair:  0


,method,rows
0,observed,76317
1,fixture_opponent_inference,19852


### Team-field reconstruction finding

The `team` field is structurally missing for all rows in the 2016-17
and 2017-18 seasons:

- 2016-17: 8,567 / 8,567 rows missing
- 2017-18: 11,285 / 11,285 rows missing
- Total affected: 19,852 rows

The missing values were not treated as random missing data and were not
imputed using player history.

Instead, the player's team was reconstructed deterministically from the
fixture structure. Each `(season, fixture)` contains exactly two unique
`opponent_team` identifiers. For each player row, the team is therefore
the other team participating in that fixture.

Validation results:

- 1,900 / 1,900 fixtures contained exactly two teams.
- 0 `opponent_team` values were missing.
- 19,852 / 19,852 missing team values were recoverable.
- The reconstruction rule was tested against all 76,317 rows from
  seasons where `team` was already observed.
- Validation accuracy was 100%.
- Observed-vs-inferred mismatches: 0.

The reconstructed team names use a season-specific
`(season, team_id) -> team_name` mapping derived from
`opponent_team` and `opp_team_name`.

Therefore fixture-based reconstruction is accepted as the canonical
repair for the historical missing `team` values.

## 6. Normalize positions, but do not repair history by player name

The reference notebook correctly observes `GK`/`GKP` inconsistency, but then hardcodes several historical player-position corrections by name.

For the canonical dataset:

- normalize spelling (`GKP -> GK`);
- validate allowed categories;
- audit position changes by player and season;
- source historical positions from authoritative player-season data if corrections are required.

In [19]:
# ============================================================
# STEP 6A — NORMALIZE POSITION LABELS
# ============================================================

if "position" in df_clean.columns:
    position_before = df_clean["position"].copy()

    df_clean["position"] = (
        df_clean["position"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({"GKP": "GK"})
    )

    allowed_positions = {"GK", "DEF", "MID", "FWD"}

    unexpected_positions = sorted(
        set(df_clean["position"].dropna().unique())
        - allowed_positions
    )

    print("Position counts after normalization:")
    display(
        df_clean["position"]
        .value_counts(dropna=False)
        .rename_axis("position")
        .reset_index(name="rows")
    )

    print("Unexpected positions:", unexpected_positions)

    changed = (
        position_before.astype("string")
        != df_clean["position"].astype("string")
    )

    print(
        "Rows changed by label normalization:",
        int(changed.fillna(False).sum()),
    )

Position counts after normalization:


,position,rows
0,MID,39163
1,DEF,33683
2,FWD,12669
3,GK,10654


Unexpected positions: []
Rows changed by label normalization: 101


In [20]:
# ============================================================
# STEP 6B — AUDIT WITHIN-SEASON POSITION CHANGES
# ============================================================

if {"season", "player_id", "position"}.issubset(df_clean.columns):

    position_variation = (
        df_clean
        .groupby(["season", "player_id"])["position"]
        .nunique(dropna=True)
        .sort_values(ascending=False)
    )

    suspect = position_variation[
        position_variation > 1
    ]

    print(
        "Player-seasons with >1 recorded position:",
        len(suspect),
    )

    if len(suspect):
        suspect_keys = (
            suspect
            .reset_index()[["season", "player_id"]]
        )

        details = df_clean.merge(
            suspect_keys,
            on=["season", "player_id"],
            how="inner",
        )

        cols = [
            c for c in [
                "season",
                "player_id",
                "player_name",
                "gameweek",
                "kickoff_time",
                "position",
            ]
            if c in details.columns
        ]

        display(
            details[cols]
            .sort_values(
                [
                    "season",
                    "player_id",
                    "kickoff_time",
                ]
            )
            .head(200)
        )

Player-seasons with >1 recorded position: 0


### Position audit conclusion

Position labels were normalized to the canonical FPL classes
`GK`, `DEF`, `MID`, and `FWD`.

- 101 rows were changed through label normalization (`GKP` → `GK`).
- No unexpected position labels remain.
- No player-season contains more than one recorded position.

No player-specific position overrides are required based on the available
data. Position normalization is therefore accepted as a canonical
cleaning transformation.

## 7. Audit chronology, blanks and double gameweeks

Gameweek number alone is not a safe sort key.

All temporal ordering should use `kickoff_time`, with `fixture_id` as a tie-breaker where available.

We explicitly inspect:
- multiple fixtures for a player in one GW;
- long gaps between appearances;
- same-kickoff duplicate rows;
- fixture ordering.

In [21]:
# ============================================================
# STEP 7A — CANONICAL CHRONOLOGICAL ORDER
# ============================================================

fixture_col = next(
    (c for c in ["fixture_id", "fixture"] if c in df_clean.columns),
    None,
)

sort_cols = [
    c for c in [
        "season",
        "player_id",
        "kickoff_time",
        fixture_col,
    ]
    if c is not None and c in df_clean.columns
]

print("Chronological sort columns:", sort_cols)

df_clean = (
    df_clean
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

display(
    df_clean[
        [
            c for c in [
                "season",
                "player_id",
                "player_name",
                "gameweek",
                fixture_col,
                "kickoff_time",
            ]
            if c is not None and c in df_clean.columns
        ]
    ].head(20)
)

Chronological sort columns: ['season', 'player_id', 'kickoff_time', 'fixture']


,season,player_id,player_name,gameweek,fixture,kickoff_time
0,2016-17,6,Héctor Bellerín,1,8,2016-08-14 15:00:00+00:00
1,2016-17,6,Héctor Bellerín,2,13,2016-08-20 16:30:00+00:00
2,2016-17,6,Héctor Bellerín,3,28,2016-08-27 14:00:00+00:00
3,2016-17,6,Héctor Bellerín,4,31,2016-09-10 14:00:00+00:00
4,2016-17,6,Héctor Bellerín,5,43,2016-09-17 14:00:00+00:00
5,2016-17,6,Héctor Bellerín,6,51,2016-09-24 16:30:00+00:00
6,2016-17,6,Héctor Bellerín,7,61,2016-10-02 15:30:00+00:00
7,2016-17,6,Héctor Bellerín,8,71,2016-10-15 14:00:00+00:00
8,2016-17,6,Héctor Bellerín,9,81,2016-10-22 14:00:00+00:00
9,2016-17,6,Héctor Bellerín,10,97,2016-10-29 11:30:00+00:00


In [22]:
# ============================================================
# STEP 7B — AUDIT MULTIPLE FIXTURES WITHIN A GAMEWEEK
# ============================================================

required = {
    "season",
    "player_id",
    "gameweek",
}

if required.issubset(df_clean.columns):

    appearances_per_gw = (
        df_clean
        .groupby(
            ["season", "player_id", "gameweek"]
        )
        .size()
        .rename("n_fixtures")
        .reset_index()
    )

    dgw_rows = appearances_per_gw[
        appearances_per_gw["n_fixtures"] > 1
    ].copy()

    print(
        "Player-gameweeks with >1 recorded fixture:",
        len(dgw_rows),
    )

    display(
        dgw_rows
        .sort_values(
            ["season", "gameweek", "player_id"]
        )
        .head(100)
    )

Player-gameweeks with >1 recorded fixture: 5790


,season,player_id,gameweek,n_fixtures
2069,2016-17,128,27,2
3345,2016-17,218,27,2
3381,2016-17,219,27,2
3417,2016-17,226,27,2
3453,2016-17,227,27,2
3489,2016-17,228,27,2
3525,2016-17,233,27,2
3561,2016-17,235,27,2
3597,2016-17,236,27,2
3633,2016-17,238,27,2


In [23]:
if len(dgw_rows):

    dgw_examples = (
        df_clean.merge(
            dgw_rows[
                ["season", "player_id", "gameweek"]
            ],
            on=[
                "season",
                "player_id",
                "gameweek",
            ],
            how="inner",
        )
    )

    cols = [
        c for c in [
            "season",
            "gameweek",
            "player_id",
            "player_name",
            fixture_col,
            "kickoff_time",
            "team",
            "opp_team_name",
            "was_home",
            "minutes",
            "total_points",
        ]
        if c is not None and c in dgw_examples.columns
    ]

    display(
        dgw_examples[cols]
        .sort_values(
            [
                "season",
                "player_id",
                "gameweek",
                "kickoff_time",
            ]
        )
        .head(100)
    )

,season,gameweek,player_id,player_name,fixture,kickoff_time,team,opp_team_name,was_home,minutes,total_points
0,2016-17,36,6,Héctor Bellerín,351,2017-05-07 15:00:00+00:00,Arsenal,Man Utd,True,7,1
1,2016-17,36,6,Héctor Bellerín,257,2017-05-10 18:45:00+00:00,Arsenal,Southampton,False,55,0
2,2016-17,37,6,Héctor Bellerín,366,2017-05-13 16:30:00+00:00,Arsenal,Stoke,False,90,9
3,2016-17,37,6,Héctor Bellerín,331,2017-05-16 18:45:00+00:00,Arsenal,Sunderland,True,90,5
4,2016-17,36,7,Kieran Gibbs,351,2017-05-07 15:00:00+00:00,Arsenal,Man Utd,True,90,6
5,2016-17,36,7,Kieran Gibbs,257,2017-05-10 18:45:00+00:00,Arsenal,Southampton,False,90,7
6,2016-17,37,7,Kieran Gibbs,366,2017-05-13 16:30:00+00:00,Arsenal,Stoke,False,0,0
7,2016-17,37,7,Kieran Gibbs,331,2017-05-16 18:45:00+00:00,Arsenal,Sunderland,True,68,6
8,2016-17,34,11,Calum Chambers,332,2017-04-22 14:00:00+00:00,Middlesbrough,Bournemouth,False,90,0
9,2016-17,34,11,Calum Chambers,278,2017-04-26 18:45:00+00:00,Middlesbrough,Sunderland,True,90,6


In [24]:
# ============================================================
# STEP 7C — AUDIT TIME GAPS BETWEEN PLAYER RECORDS
# ============================================================

if {
    "season",
    "player_id",
    "kickoff_time",
}.issubset(df_clean.columns):

    temp = (
        df_clean
        .sort_values(
            [
                "season",
                "player_id",
                "kickoff_time",
            ]
        )
        .copy()
    )

    temp["days_since_previous_recorded_fixture"] = (
        temp
        .groupby(
            ["season", "player_id"]
        )["kickoff_time"]
        .diff()
        .dt.total_seconds()
        .div(86400)
    )

    print("Gap distribution:")
    display(
        temp[
            "days_since_previous_recorded_fixture"
        ].describe(
            percentiles=[
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
    )


    long_gaps = temp[
        temp[
            "days_since_previous_recorded_fixture"
        ] > 30
    ]

    print(
        "\nPlayer records following >30-day gaps:",
        len(long_gaps),
    )

    cols = [
        c for c in [
            "season",
            "player_id",
            "player_name",
            "gameweek",
            fixture_col,
            "kickoff_time",
            "days_since_previous_recorded_fixture",
        ]
        if c is not None and c in long_gaps.columns
    ]

    display(
        long_gaps[cols]
        .sort_values(
            "days_since_previous_recorded_fixture",
            ascending=False,
        )
        .head(100)
    )

Gap distribution:


,days_since_previous_recorded_fixture
count,93383.000000
mean,7.422057
std,4.960819
min,1.687500
50%,6.913194
75%,7.989583
90%,13.895833
95%,14.968750
99%,22.979167
max,46.312500



Player records following >30-day gaps: 667


,season,player_id,player_name,gameweek,fixture,kickoff_time,days_since_previous_recorded_fixture
81000,2022-23,299,Kyle Walker,17,167,2022-12-28 20:00:00+00:00,46.312500
81038,2022-23,300,Ilkay Gündogan,17,167,2022-12-28 20:00:00+00:00,46.312500
81076,2022-23,301,Kevin De Bruyne,17,167,2022-12-28 20:00:00+00:00,46.312500
81114,2022-23,302,John Stones,17,167,2022-12-28 20:00:00+00:00,46.312500
81304,2022-23,307,Ederson Santana de Moraes,17,167,2022-12-28 20:00:00+00:00,46.312500
81266,2022-23,306,João Cancelo,17,167,2022-12-28 20:00:00+00:00,46.312500
81228,2022-23,305,Jack Grealish,17,167,2022-12-28 20:00:00+00:00,46.312500
81152,2022-23,303,Riyad Mahrez,17,167,2022-12-28 20:00:00+00:00,46.312500
92383,2022-23,600,Ben Knight,17,167,2022-12-28 20:00:00+00:00,46.312500
91411,2022-23,573,Rico Lewis,17,167,2022-12-28 20:00:00+00:00,46.312500


In [25]:
# ============================================================
# STEP 7D — AUDIT DUPLICATE PLAYER-FIXTURE ROWS
# ============================================================

if fixture_col is not None:

    duplicate_key = [
        "season",
        "player_id",
        fixture_col,
    ]

    duplicate_player_fixture = (
        df_clean[
            df_clean.duplicated(
                duplicate_key,
                keep=False,
            )
        ]
        .sort_values(duplicate_key)
    )

    print(
        "Rows participating in duplicate "
        "player-fixture keys:",
        len(duplicate_player_fixture),
    )

    display(
        duplicate_player_fixture.head(100)
    )

Rows participating in duplicate player-fixture keys: 0


,season,player_name,position,team,assists,bonus,bps,clean_sheets,creativity,player_id,fixture,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,opponent_team,opp_team_name,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,gameweek,team_recovery_method


In [26]:
same_kickoff_key = [
    "season",
    "player_id",
    "kickoff_time",
]

same_kickoff = (
    df_clean[
        df_clean.duplicated(
            same_kickoff_key,
            keep=False,
        )
    ]
    .sort_values(same_kickoff_key)
)

print(
    "Rows participating in repeated "
    "player/kickoff-time keys:",
    len(same_kickoff),
)

display(
    same_kickoff[
        [
            c for c in [
                "season",
                "player_id",
                "player_name",
                "gameweek",
                fixture_col,
                "kickoff_time",
                "team",
                "opp_team_name",
                "total_points",
            ]
            if c is not None and c in same_kickoff.columns
        ]
    ].head(100)
)

Rows participating in repeated player/kickoff-time keys: 0


,season,player_id,player_name,gameweek,fixture,kickoff_time,team,opp_team_name,total_points


In [27]:
# ============================================================
# STEP 7E — VALIDATE PLAYER CHRONOLOGY
# ============================================================

chronology = (
    df_clean
    .sort_values(sort_cols)
    .copy()
)

chronology["_previous_kickoff"] = (
    chronology
    .groupby(
        ["season", "player_id"]
    )["kickoff_time"]
    .shift(1)
)

chronology["_chronology_violation"] = (
    chronology["_previous_kickoff"].notna()
    & (
        chronology["kickoff_time"]
        < chronology["_previous_kickoff"]
    )
)

n_violations = int(
    chronology["_chronology_violation"].sum()
)

print(
    "Player chronology violations:",
    n_violations,
)

assert n_violations == 0

Player chronology violations: 0


### Chronology and fixture-granularity conclusion

The dataset is structurally consistent at player-fixture level.

- 5,790 player-gameweeks contain more than one recorded fixture and are
  retained as legitimate multi-fixture / Double Gameweek observations.
- No duplicate `(season, player_id, fixture)` keys were found.
- No repeated `(season, player_id, kickoff_time)` keys were found.
- No player-level chronological ordering violations were found.
- The median interval between consecutive recorded fixtures is approximately
  6.9 days.
- Long gaps occur, but are plausible for injuries, suspensions, selection
  changes, transfers, postponements, or other periods without a recorded
  fixture. They are not removed.

All downstream temporal operations must therefore use `kickoff_time`
(with fixture identifier as a tie-breaker) rather than gameweek alone.

## 8. Validate match/team consistency

For each fixture, the dataset should describe two opposing teams consistently.

Where sufficient identifiers exist, validate:
- the same fixture does not map to multiple opponent pairs;
- home/away flags are consistent;
- player team differs from opponent;
- team and opponent score relationships are coherent.

In [28]:
# ============================================================
# STEP 8A — VALIDATE TEAM != OPPONENT
# ============================================================

required = {
    "team",
    "opp_team_name",
}

if required.issubset(df_clean.columns):

    team_name = (
        df_clean["team"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    opponent_name = (
        df_clean["opp_team_name"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    same_team = (
        team_name.notna()
        & opponent_name.notna()
        & (team_name == opponent_name)
    )

    print(
        "Rows where team == opponent:",
        int(same_team.sum()),
    )

    if same_team.any():
        display(
            df_clean.loc[
                same_team,
                [
                    c for c in [
                        "season",
                        "gameweek",
                        "fixture",
                        "player_id",
                        "player_name",
                        "team",
                        "opp_team_name",
                        "was_home",
                    ]
                    if c in df_clean.columns
                ],
            ].head(100)
        )

Rows where team == opponent: 0


In [29]:
# ============================================================
# STEP 8B — VALIDATE TWO TEAMS PER FIXTURE
# ============================================================

fixture_col = next(
    (c for c in ["fixture_id", "fixture"] if c in df_clean.columns),
    None,
)

if fixture_col is not None:

    fixture_team_counts = (
        df_clean
        .groupby(["season", fixture_col])
        .agg(
            n_teams=("team", "nunique"),
            n_opponents=("opp_team_name", "nunique"),
            rows=("player_id", "size"),
        )
        .reset_index()
    )

    invalid_fixture_teams = fixture_team_counts[
        (fixture_team_counts["n_teams"] != 2)
        | (fixture_team_counts["n_opponents"] != 2)
    ]

    print(
        "Fixtures not resolving to exactly two teams:",
        len(invalid_fixture_teams),
    )

    display(invalid_fixture_teams.head(100))

Fixtures not resolving to exactly two teams: 0


,season,fixture,n_teams,n_opponents,rows


In [30]:
# ============================================================
# STEP 8C — VALIDATE RECIPROCAL TEAM/OPPONENT PAIRS
# ============================================================

fixture_pairs = (
    df_clean[
        [
            "season",
            fixture_col,
            "team",
            "opp_team_name",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["season", fixture_col, "team"]
    )
)

pair_counts = (
    fixture_pairs
    .groupby(["season", fixture_col])
    .size()
    .rename("n_team_opponent_pairs")
    .reset_index()
)

bad_pairs = pair_counts[
    pair_counts["n_team_opponent_pairs"] != 2
]

print(
    "Fixtures without exactly two reciprocal pairs:",
    len(bad_pairs),
)

display(bad_pairs.head(100))

Fixtures without exactly two reciprocal pairs: 0


,season,fixture,n_team_opponent_pairs


In [31]:
# ============================================================
# STEP 8D — VALIDATE HOME/AWAY CONSISTENCY
# ============================================================

team_fixture_home = (
    df_clean
    .groupby(
        [
            "season",
            fixture_col,
            "team",
        ],
        as_index=False,
    )
    .agg(
        n_home_values=(
            "was_home",
            lambda s: s.dropna().nunique()
        ),
        home_values=(
            "was_home",
            lambda s: tuple(
                sorted(s.dropna().unique())
            )
        ),
    )
)

inconsistent_team_home = team_fixture_home[
    team_fixture_home["n_home_values"] != 1
]

print(
    "Team-fixtures with inconsistent was_home:",
    len(inconsistent_team_home),
)

display(inconsistent_team_home.head(100))

Team-fixtures with inconsistent was_home: 0


,season,fixture,team,n_home_values,home_values


In [32]:
fixture_home_structure = (
    team_fixture_home
    .groupby(
        ["season", fixture_col]
    )
    .agg(
        n_teams=("team", "size"),
        home_flags=(
            "home_values",
            lambda s: tuple(sorted(
                v[0]
                for v in s
                if len(v) == 1
            ))
        ),
    )
    .reset_index()
)

bad_home_structure = fixture_home_structure[
    (fixture_home_structure["n_teams"] != 2)
    | (
        fixture_home_structure["home_flags"]
        != (False, True)
    )
]

print(
    "Fixtures with invalid home/away structure:",
    len(bad_home_structure),
)

display(bad_home_structure.head(100))

Fixtures with invalid home/away structure: 0


,season,fixture,n_teams,home_flags


In [33]:
score_cols = [
    c for c in [
        "team_h_score",
        "team_a_score",
        "goals_scored",
        "goals_conceded",
    ]
    if c in df_clean.columns
]

print("Available score-related columns:")
print(score_cols)

Available score-related columns:
['team_h_score', 'team_a_score', 'goals_scored', 'goals_conceded']


In [34]:
# ============================================================
# STEP 8E — VALIDATE FIXTURE SCORE CONSISTENCY
# ============================================================

score_fixture_audit = (
    df_clean
    .groupby(["season", fixture_col])
    .agg(
        n_home_scores=("team_h_score", "nunique"),
        n_away_scores=("team_a_score", "nunique"),
        home_scores=(
            "team_h_score",
            lambda s: tuple(sorted(s.dropna().unique()))
        ),
        away_scores=(
            "team_a_score",
            lambda s: tuple(sorted(s.dropna().unique()))
        ),
        rows=("player_id", "size"),
    )
    .reset_index()
)

bad_fixture_scores = score_fixture_audit[
    (score_fixture_audit["n_home_scores"] != 1)
    | (score_fixture_audit["n_away_scores"] != 1)
]

print(
    "Fixtures with inconsistent final scores:",
    len(bad_fixture_scores),
)

display(bad_fixture_scores.head(100))

Fixtures with inconsistent final scores: 0


,season,fixture,n_home_scores,n_away_scores,home_scores,away_scores,rows


In [35]:
# ============================================================
# STEP 8F — DERIVE ROW-LEVEL TEAM / OPPONENT FINAL SCORES
# ============================================================

score_check = df_clean[
    [
        "season",
        fixture_col,
        "player_id",
        "player_name",
        "team",
        "opp_team_name",
        "was_home",
        "team_h_score",
        "team_a_score",
        "goals_scored",
        "goals_conceded",
        "minutes",
    ]
].copy()

score_check["team_final_score"] = np.where(
    score_check["was_home"],
    score_check["team_h_score"],
    score_check["team_a_score"],
)

score_check["opponent_final_score"] = np.where(
    score_check["was_home"],
    score_check["team_a_score"],
    score_check["team_h_score"],
)

display(score_check.head(20))

,season,fixture,player_id,player_name,team,opp_team_name,was_home,team_h_score,team_a_score,goals_scored,goals_conceded,minutes,team_final_score,opponent_final_score
0,2016-17,8,6,Héctor Bellerín,Arsenal,Liverpool,True,3.0,4.0,0,4,90,3.0,4.0
1,2016-17,13,6,Héctor Bellerín,Arsenal,Leicester,False,0.0,0.0,0,0,90,0.0,0.0
2,2016-17,28,6,Héctor Bellerín,Arsenal,Watford,False,1.0,3.0,0,1,90,3.0,1.0
3,2016-17,31,6,Héctor Bellerín,Arsenal,Southampton,True,2.0,1.0,0,1,90,2.0,1.0
4,2016-17,43,6,Héctor Bellerín,Arsenal,Hull,False,1.0,4.0,0,1,90,4.0,1.0
5,2016-17,51,6,Héctor Bellerín,Arsenal,Chelsea,True,3.0,0.0,0,0,90,3.0,0.0
6,2016-17,61,6,Héctor Bellerín,Arsenal,Burnley,False,0.0,1.0,0,0,90,1.0,0.0
7,2016-17,71,6,Héctor Bellerín,Arsenal,Swansea,True,3.0,2.0,0,2,90,3.0,2.0
8,2016-17,81,6,Héctor Bellerín,Arsenal,Middlesbrough,True,0.0,0.0,0,0,90,0.0,0.0
9,2016-17,97,6,Héctor Bellerín,Arsenal,Sunderland,False,1.0,4.0,0,1,90,4.0,1.0


In [36]:
# A player's recorded goals cannot exceed their team's final goals.
player_goals_exceed_team = (
    score_check["goals_scored"]
    > score_check["team_final_score"]
)

print(
    "Rows where player goals exceed team final score:",
    int(player_goals_exceed_team.sum()),
)


# A player cannot have conceded more goals while on the pitch
# than the opponent scored in the entire match.
player_conceded_exceed_final = (
    score_check["goals_conceded"]
    > score_check["opponent_final_score"]
)

print(
    "Rows where player goals_conceded exceed opponent final score:",
    int(player_conceded_exceed_final.sum()),
)

if player_goals_exceed_team.any():
    display(
        score_check.loc[player_goals_exceed_team].head(100)
    )

if player_conceded_exceed_final.any():
    display(
        score_check.loc[player_conceded_exceed_final].head(100)
    )

Rows where player goals exceed team final score: 0
Rows where player goals_conceded exceed opponent final score: 0


In [37]:
# ============================================================
# STEP 8G — VALIDATE RECIPROCAL SCORE VIEW
# ============================================================

team_score_view = (
    score_check[
        [
            "season",
            fixture_col,
            "team",
            "opp_team_name",
            "was_home",
            "team_final_score",
            "opponent_final_score",
        ]
    ]
    .drop_duplicates()
)

score_pair_counts = (
    team_score_view
    .groupby(["season", fixture_col])
    .size()
    .rename("n_score_views")
    .reset_index()
)

bad_score_views = score_pair_counts[
    score_pair_counts["n_score_views"] != 2
]

print(
    "Fixtures without exactly two team score views:",
    len(bad_score_views),
)

Fixtures without exactly two team score views: 0


### Match/team consistency conclusion

Fixture and team identity fields are internally consistent.

- No row has the same team and opponent.
- Every fixture resolves to exactly two teams.
- Every fixture contains exactly two reciprocal team/opponent pairs.
- Home/away status is internally consistent within each team-fixture.
- Every fixture contains one home and one away team.
- Final home and away scores are constant within each fixture.
- Player goals do not exceed their team's final score.
- Player-level goals conceded do not exceed the opponent's final score.

No match/team consistency repairs are required.

## 9. FPL points reconciliation

Keep the reference notebook's scoring-reconstruction idea, but use it as a **data-quality assertion**, not as an automatic position-correction mechanism.

The helper below reconstructs the components that are available in the dataset. It should be adapted if the historical scoring rules or source schema differ.

Important: bonus points are already represented by `bonus`; do not recompute BPS-to-bonus allocation from individual rows.

In [38]:
# ============================================================
# STEP 9 — FPL POINTS RECONCILIATION
# ============================================================

def reconstruct_fpl_points(row):
    position = row.get("position")
    minutes = row.get("minutes", 0) or 0

    points = 0

    # --------------------------------------------------------
    # Appearance
    # --------------------------------------------------------
    if minutes > 0:
        points += 1

    if minutes >= 60:
        points += 1

    # --------------------------------------------------------
    # Goals
    # --------------------------------------------------------
    goal_points = {
        "GK": 6,
        "DEF": 6,
        "MID": 5,
        "FWD": 4,
    }

    goals_scored = row.get("goals_scored", 0) or 0
    points += goals_scored * goal_points.get(position, 0)

    # --------------------------------------------------------
    # Assists
    # --------------------------------------------------------
    assists = row.get("assists", 0) or 0
    points += assists * 3

    # --------------------------------------------------------
    # Clean sheets
    #
    # Clean-sheet points require at least 60 minutes.
    # --------------------------------------------------------
    clean_sheet = row.get("clean_sheets", 0) or 0

    if minutes >= 60 and clean_sheet:
        points += {
            "GK": 4,
            "DEF": 4,
            "MID": 1,
            "FWD": 0,
        }.get(position, 0)

    # --------------------------------------------------------
    # Goals conceded
    #
    # GK / DEF lose 1 point for every 2 goals conceded.
    # This deduction does NOT require 60 minutes played.
    # --------------------------------------------------------
    goals_conceded = row.get("goals_conceded", 0) or 0

    if position in {"GK", "DEF"}:
        points -= int(goals_conceded // 2)

    # --------------------------------------------------------
    # Saves
    # --------------------------------------------------------
    saves = row.get("saves", 0) or 0

    if position == "GK":
        points += int(saves // 3)

    # --------------------------------------------------------
    # Penalties
    # --------------------------------------------------------
    penalties_saved = row.get("penalties_saved", 0) or 0
    penalties_missed = row.get("penalties_missed", 0) or 0

    points += penalties_saved * 5
    points -= penalties_missed * 2

    # --------------------------------------------------------
    # Cards / own goals
    # --------------------------------------------------------
    yellow_cards = row.get("yellow_cards", 0) or 0
    red_cards = row.get("red_cards", 0) or 0
    own_goals = row.get("own_goals", 0) or 0

    points -= yellow_cards
    points -= red_cards * 3
    points -= own_goals * 2

    # --------------------------------------------------------
    # Bonus
    # --------------------------------------------------------
    bonus = row.get("bonus", 0) or 0
    points += bonus

    return points


required_for_reconstruction = {
    "position",
    "minutes",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "saves",
    "penalties_saved",
    "penalties_missed",
    "yellow_cards",
    "red_cards",
    "own_goals",
    "bonus",
    "total_points",
}


if required_for_reconstruction.issubset(df_clean.columns):

    score_check = df_clean.copy()

    score_check["reconstructed_points"] = (
        score_check.apply(
            reconstruct_fpl_points,
            axis=1,
        )
    )

    score_check["points_delta"] = (
        score_check["total_points"]
        - score_check["reconstructed_points"]
    )

    # --------------------------------------------------------
    # Overall reconciliation summary
    # --------------------------------------------------------
    print("Points-delta distribution:")
    display(
        score_check["points_delta"]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("points_delta")
        .reset_index(name="rows")
    )

    exact_match_rate = (
        score_check["points_delta"] == 0
    ).mean()

    n_exact = int(
        (score_check["points_delta"] == 0).sum()
    )

    n_mismatch = int(
        (score_check["points_delta"] != 0).sum()
    )

    print(
        f"Exact reconciliation rows: "
        f"{n_exact:,} / {len(score_check):,}"
    )

    print(
        f"Exact reconciliation rate: "
        f"{exact_match_rate:.6%}"
    )

    print(
        f"Rows with scoring mismatch: "
        f"{n_mismatch:,}"
    )

    # --------------------------------------------------------
    # Reconciliation by season
    # --------------------------------------------------------
    reconciliation_by_season = (
        score_check
        .assign(
            exact_match=(
                score_check["points_delta"] == 0
            )
        )
        .groupby("season", as_index=False)
        .agg(
            rows=("player_id", "size"),
            exact_matches=("exact_match", "sum"),
            mismatches=(
                "exact_match",
                lambda s: (~s).sum(),
            ),
            exact_match_rate=(
                "exact_match",
                "mean",
            ),
        )
    )

    reconciliation_by_season[
        "exact_match_pct"
    ] = (
        reconciliation_by_season[
            "exact_match_rate"
        ] * 100
    )

    print("\nReconciliation by season:")
    display(reconciliation_by_season)

    # --------------------------------------------------------
    # Inspect mismatches
    # --------------------------------------------------------
    mismatch_rows = score_check[
        score_check["points_delta"] != 0
    ].copy()

    mismatch_cols = [
        c for c in [
            "season",
            "player_id",
            "player_name",
            "position",
            "gameweek",
            "fixture",
            "kickoff_time",
            "minutes",
            "goals_scored",
            "assists",
            "clean_sheets",
            "goals_conceded",
            "saves",
            "penalties_saved",
            "penalties_missed",
            "yellow_cards",
            "red_cards",
            "own_goals",
            "bonus",
            "total_points",
            "reconstructed_points",
            "points_delta",
        ]
        if c in mismatch_rows.columns
    ]

    print("\nExample mismatches:")
    display(
        mismatch_rows[mismatch_cols]
        .sort_values(
            [
                "season",
                "player_id",
                "kickoff_time",
            ]
        )
        .head(100)
    )

else:
    missing_cols = sorted(
        required_for_reconstruction
        - set(df_clean.columns)
    )

    print(
        "Scoring reconstruction skipped; "
        "missing columns:",
        missing_cols,
    )

Points-delta distribution:


,points_delta,rows
0,-5,1
1,-4,4
2,-3,88
3,-2,13
4,-1,60
5,0,95829
6,1,130
7,2,25
8,3,19


Exact reconciliation rows: 95,829 / 96,169
Exact reconciliation rate: 99.646456%
Rows with scoring mismatch: 340

Reconciliation by season:


,season,rows,exact_matches,mismatches,exact_match_rate,exact_match_pct
0,2016-17,8567,8423,144,0.983191,98.319132
1,2017-18,11285,11089,196,0.982632,98.263181
2,2020-21,24365,24365,0,1.000000,100.000000
3,2021-22,25447,25447,0,1.000000,100.000000
4,2022-23,26505,26505,0,1.000000,100.000000



Example mismatches:


,season,player_id,player_name,position,gameweek,fixture,kickoff_time,minutes,goals_scored,assists,clean_sheets,goals_conceded,saves,penalties_saved,penalties_missed,yellow_cards,red_cards,own_goals,bonus,total_points,reconstructed_points,points_delta
800,2016-17,48,Joshua King,FWD,3,22,2016-08-27 14:00:00+00:00,90,1,0,0,1,0,0,0,0,0,0,2,9,8,1
801,2016-17,48,Joshua King,FWD,4,32,2016-09-10 14:00:00+00:00,73,0,0,1,0,0,0,0,0,0,0,0,3,2,1
804,2016-17,48,Joshua King,FWD,7,69,2016-10-01 14:00:00+00:00,29,1,0,0,1,0,0,0,0,0,0,0,6,5,1
806,2016-17,48,Joshua King,FWD,9,82,2016-10-22 11:30:00+00:00,87,0,0,1,0,0,0,0,0,0,0,0,3,2,1
809,2016-17,48,Joshua King,FWD,12,116,2016-11-19 15:00:00+00:00,80,0,0,1,0,0,0,0,0,0,0,0,3,2,1
813,2016-17,48,Joshua King,FWD,16,151,2016-12-13 19:45:00+00:00,78,0,0,1,0,0,0,0,0,0,0,0,3,2,1
816,2016-17,48,Joshua King,FWD,19,189,2016-12-31 15:00:00+00:00,18,1,0,0,0,0,0,0,0,0,0,0,6,5,1
817,2016-17,48,Joshua King,FWD,20,191,2017-01-03 19:45:00+00:00,62,0,0,1,0,0,0,0,0,0,0,0,3,2,1
819,2016-17,48,Joshua King,FWD,22,212,2017-01-21 15:00:00+00:00,73,1,0,0,2,0,0,0,0,0,0,0,7,6,1
821,2016-17,48,Joshua King,FWD,24,233,2017-02-04 15:00:00+00:00,90,2,0,0,6,0,0,0,0,0,0,1,13,11,2


In [39]:
if "mismatch_rows" in globals() and len(mismatch_rows):

    print("Mismatches by position:")
    display(
        mismatch_rows["position"]
        .value_counts()
        .rename_axis("position")
        .reset_index(name="rows")
    )

    print("\nMismatches by minutes played:")
    display(
        mismatch_rows["minutes"]
        .value_counts()
        .sort_index()
        .rename_axis("minutes")
        .reset_index(name="rows")
        .head(100)
    )

    print("\nMismatches by points delta:")
    display(
        mismatch_rows["points_delta"]
        .value_counts()
        .sort_index()
        .rename_axis("points_delta")
        .reset_index(name="rows")
    )

Mismatches by position:


,position,rows
0,DEF,147
1,FWD,104
2,MID,89



Mismatches by minutes played:


,minutes,rows
0,2,1
1,10,1
2,11,1
3,15,1
4,17,1
5,18,2
6,20,2
7,23,1
8,24,1
9,27,1



Mismatches by points delta:


,points_delta,rows
0,-5,1
1,-4,4
2,-3,88
3,-2,13
4,-1,60
5,1,130
6,2,25
7,3,19


In [40]:
# ============================================================
# STEP 9A — LOCALIZE SCORING MISMATCHES
# ============================================================

print("Mismatches by season:")
display(
    mismatch_rows
    .groupby("season")
    .size()
    .rename("rows")
    .reset_index()
)

print("\nMismatches by season and position:")
display(
    mismatch_rows
    .groupby(["season", "position"])
    .size()
    .rename("rows")
    .reset_index()
)

print("\nMismatch rate by season:")
season_reconciliation = (
    score_check
    .assign(mismatch=score_check["points_delta"] != 0)
    .groupby("season", as_index=False)
    .agg(
        rows=("player_id", "size"),
        mismatches=("mismatch", "sum"),
        mismatch_rate=("mismatch", "mean"),
    )
)

season_reconciliation["mismatch_pct"] = (
    season_reconciliation["mismatch_rate"] * 100
)

display(season_reconciliation)

Mismatches by season:


,season,rows
0,2016-17,144
1,2017-18,196



Mismatches by season and position:


,season,position,rows
0,2016-17,DEF,57
1,2016-17,FWD,69
2,2016-17,MID,18
3,2017-18,DEF,90
4,2017-18,FWD,35
5,2017-18,MID,71



Mismatch rate by season:


,season,rows,mismatches,mismatch_rate,mismatch_pct
0,2016-17,8567,144,0.016809,1.680868
1,2017-18,11285,196,0.017368,1.736819
2,2020-21,24365,0,0.000000,0.000000
3,2021-22,25447,0,0.000000,0.000000
4,2022-23,26505,0,0.000000,0.000000


In [42]:
# ============================================================
# STEP 9B — MISMATCHES BY POSITION-SENSITIVE COMPONENTS
# ============================================================

component_audit = (
    mismatch_rows
    .groupby(
        [
            "position",
            "goals_scored",
            "clean_sheets",
        ]
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(component_audit.head(50))

,position,goals_scored,clean_sheets,rows
1,DEF,0,1,85
0,DEF,0,0,53
6,FWD,0,1,49
12,MID,0,1,37
7,FWD,1,0,35
11,MID,0,0,26
13,MID,1,0,16
8,FWD,1,1,14
14,MID,1,1,6
9,FWD,2,0,5


In [43]:
print(
    "Mismatches with >=1 goal:",
    int((mismatch_rows["goals_scored"] > 0).sum()),
)

print(
    "Mismatches with clean sheet:",
    int((mismatch_rows["clean_sheets"] > 0).sum()),
)

print(
    "Mismatches with neither goal nor clean sheet:",
    int(
        (
            (mismatch_rows["goals_scored"] == 0)
            & (mismatch_rows["clean_sheets"] == 0)
        ).sum()
    ),
)

Mismatches with >=1 goal: 90
Mismatches with clean sheet: 199
Mismatches with neither goal nor clean sheet: 79


In [44]:
# ============================================================
# STEP 9C — IDENTIFY PLAYER-SEASONS DRIVING MISMATCHES
# ============================================================

player_season_mismatch = (
    mismatch_rows
    .groupby(
        [
            "season",
            "player_id",
            "player_name",
            "position",
        ],
        as_index=False,
    )
    .agg(
        mismatch_rows=("points_delta", "size"),
        delta_values=(
            "points_delta",
            lambda s: tuple(sorted(set(s)))
        ),
        min_delta=("points_delta", "min"),
        max_delta=("points_delta", "max"),
        goals=("goals_scored", "sum"),
        clean_sheet_rows=("clean_sheets", "sum"),
    )
    .sort_values(
        "mismatch_rows",
        ascending=False,
    )
)

display(player_season_mismatch.head(100))

,season,player_id,player_name,position,mismatch_rows,delta_values,min_delta,max_delta,goals,clean_sheet_rows
21,2017-18,384,Eric Dier,DEF,23,"(-3, 1, 2)",-3,2,0,15
10,2016-17,390,Eric Dier,DEF,21,"(-4, -3, 1)",-4,1,2,16
0,2016-17,48,Joshua King,FWD,20,"(1, 2, 3)",1,3,16,9
4,2016-17,209,Roberto Firmino,FWD,19,"(1, 2)",1,2,11,11
16,2017-18,250,Fernando Luiz Rosa,DEF,19,"(-4, -3, -1, 1, 2)",-4,2,3,15
17,2017-18,277,Ashley Young,DEF,17,"(-3, -1, 1)",-3,1,2,9
5,2016-17,233,Fernando Luiz Rosa,DEF,16,"(-4, -3, -1, 1)",-4,1,2,10
19,2017-18,284,Marcus Rashford,MID,15,"(-2, -1)",-2,-1,7,10
3,2016-17,182,Daniel Amartey,DEF,14,"(-3, 1, 2, 3)",-3,3,0,4
18,2017-18,280,Anthony Martial,FWD,14,"(1, 2)",1,2,9,9


In [45]:
# ============================================================
# STEP 9D — TEST ALTERNATIVE POSITIONS
# AUDIT ONLY — DO NOT MODIFY df_clean
# ============================================================

VALID_POSITIONS = ["GK", "DEF", "MID", "FWD"]


def score_under_position(row, candidate_position):
    test_row = row.copy()
    test_row["position"] = candidate_position
    return reconstruct_fpl_points(test_row)


position_test = df_clean.copy()

for pos in VALID_POSITIONS:
    position_test[f"score_as_{pos}"] = position_test.apply(
        lambda row: score_under_position(row, pos),
        axis=1,
    )

    position_test[f"matches_as_{pos}"] = (
        position_test[f"score_as_{pos}"]
        == position_test["total_points"]
    )

In [46]:
# ============================================================
# STEP 9E — PLAYER-SEASON POSITION EVIDENCE
# ============================================================

records = []

for (
    season,
    player_id,
    player_name,
    recorded_position,
), group in position_test.groupby(
    [
        "season",
        "player_id",
        "player_name",
        "position",
    ]
):

    record = {
        "season": season,
        "player_id": player_id,
        "player_name": player_name,
        "recorded_position": recorded_position,
        "rows": len(group),
    }

    for pos in VALID_POSITIONS:
        matches = int(group[f"matches_as_{pos}"].sum())

        record[f"{pos}_matches"] = matches
        record[f"{pos}_match_pct"] = (
            matches / len(group) * 100
        )

    records.append(record)


position_evidence = pd.DataFrame(records)

In [47]:
# ============================================================
# STEP 9E — PLAYER-SEASON POSITION EVIDENCE
# ============================================================

records = []

for (
    season,
    player_id,
    player_name,
    recorded_position,
), group in position_test.groupby(
    [
        "season",
        "player_id",
        "player_name",
        "position",
    ]
):

    record = {
        "season": season,
        "player_id": player_id,
        "player_name": player_name,
        "recorded_position": recorded_position,
        "rows": len(group),
    }

    for pos in VALID_POSITIONS:
        matches = int(group[f"matches_as_{pos}"].sum())

        record[f"{pos}_matches"] = matches
        record[f"{pos}_match_pct"] = (
            matches / len(group) * 100
        )

    records.append(record)


position_evidence = pd.DataFrame(records)

In [48]:
match_cols = [
    f"{pos}_matches"
    for pos in VALID_POSITIONS
]

position_evidence["best_match_count"] = (
    position_evidence[match_cols].max(axis=1)
)


def get_best_positions(row):
    best = row["best_match_count"]

    return tuple(
        pos
        for pos in VALID_POSITIONS
        if row[f"{pos}_matches"] == best
    )


position_evidence["best_positions"] = (
    position_evidence.apply(
        get_best_positions,
        axis=1,
    )
)

position_evidence["n_best_positions"] = (
    position_evidence["best_positions"].apply(len)
)

position_evidence["best_position"] = (
    position_evidence["best_positions"].apply(
        lambda x: x[0] if len(x) == 1 else pd.NA
    )
)

position_evidence[
    "best_position_match_pct"
] = (
    position_evidence["best_match_count"]
    / position_evidence["rows"]
    * 100
)

position_evidence[
    "position_change_suggested"
] = (
    position_evidence["n_best_positions"].eq(1)
    & position_evidence["best_position"].notna()
    & (
        position_evidence["best_position"]
        != position_evidence["recorded_position"]
    )
)

In [49]:
suggested_position_changes = (
    position_evidence[
        position_evidence[
            "position_change_suggested"
        ]
    ]
    .sort_values(
        [
            "best_position_match_pct",
            "rows",
        ],
        ascending=[False, False],
    )
)

display(
    suggested_position_changes[
        [
            "season",
            "player_id",
            "player_name",
            "recorded_position",
            "best_position",
            "rows",
            "GK_matches",
            "DEF_matches",
            "MID_matches",
            "FWD_matches",
            "best_position_match_pct",
        ]
    ].head(100)
)

,season,player_id,player_name,recorded_position,best_position,rows,GK_matches,DEF_matches,MID_matches,FWD_matches,best_position_match_pct
21,2016-17,48,Joshua King,FWD,MID,38,10,10,38,18,100.0
73,2016-17,182,Daniel Amartey,DEF,MID,38,24,24,38,33,100.0
84,2016-17,209,Roberto Firmino,FWD,MID,38,11,11,38,19,100.0
94,2016-17,233,Fernando Luiz Rosa,DEF,MID,38,22,22,38,27,100.0
109,2016-17,260,Ashley Young,DEF,MID,38,33,33,38,34,100.0
112,2016-17,267,Anthony Martial,FWD,MID,38,25,25,38,27,100.0
113,2016-17,271,Marcus Rashford,MID,FWD,38,26,26,27,38,100.0
126,2016-17,311,Jay Rodriguez,FWD,MID,38,29,29,38,32,100.0
149,2016-17,390,Eric Dier,DEF,MID,38,17,17,38,22,100.0
174,2016-17,464,Michail Antonio,FWD,MID,38,24,24,38,25,100.0


In [50]:
high_confidence_position_changes = (
    suggested_position_changes[
        suggested_position_changes[
            "best_position_match_pct"
        ] == 100
    ]
    .copy()
)

print(
    "High-confidence player-season position corrections:",
    len(high_confidence_position_changes),
)

display(
    high_confidence_position_changes[
        [
            "season",
            "player_id",
            "player_name",
            "recorded_position",
            "best_position",
            "rows",
            "best_position_match_pct",
        ]
    ]
)

High-confidence player-season position corrections: 24


,season,player_id,player_name,recorded_position,best_position,rows,best_position_match_pct
21,2016-17,48,Joshua King,FWD,MID,38,100.0
73,2016-17,182,Daniel Amartey,DEF,MID,38,100.0
84,2016-17,209,Roberto Firmino,FWD,MID,38,100.0
94,2016-17,233,Fernando Luiz Rosa,DEF,MID,38,100.0
109,2016-17,260,Ashley Young,DEF,MID,38,100.0
112,2016-17,267,Anthony Martial,FWD,MID,38,100.0
113,2016-17,271,Marcus Rashford,MID,FWD,38,100.0
126,2016-17,311,Jay Rodriguez,FWD,MID,38,100.0
149,2016-17,390,Eric Dier,DEF,MID,38,100.0
174,2016-17,464,Michail Antonio,FWD,MID,38,100.0


In [51]:
# ============================================================
# STEP 9F — VALIDATE PROPOSED POSITION CORRECTIONS
# ============================================================

df_position_candidate = df_clean.copy()

correction_lookup = {
    (row.season, row.player_id): row.best_position
    for row in high_confidence_position_changes.itertuples()
}

candidate_keys = list(
    zip(
        df_position_candidate["season"],
        df_position_candidate["player_id"],
    )
)

df_position_candidate["_candidate_position"] = [
    correction_lookup.get(key, pd.NA)
    for key in candidate_keys
]

change_mask = (
    df_position_candidate[
        "_candidate_position"
    ].notna()
)

df_position_candidate.loc[
    change_mask,
    "position",
] = df_position_candidate.loc[
    change_mask,
    "_candidate_position",
]

In [52]:
df_position_candidate[
    "reconstructed_points"
] = (
    df_position_candidate.apply(
        reconstruct_fpl_points,
        axis=1,
    )
)

df_position_candidate[
    "points_delta"
] = (
    df_position_candidate["total_points"]
    - df_position_candidate[
        "reconstructed_points"
    ]
)

candidate_mismatches = (
    df_position_candidate[
        "points_delta"
    ] != 0
)

print(
    "Mismatches before position correction:",
    340,
)

print(
    "Mismatches after high-confidence corrections:",
    int(candidate_mismatches.sum()),
)

print(
    "Reconciliation after corrections:",
    f"{(~candidate_mismatches).mean():.6%}",
)

display(
    df_position_candidate.loc[
        candidate_mismatches,
        [
            c for c in [
                "season",
                "player_id",
                "player_name",
                "position",
                "gameweek",
                "minutes",
                "total_points",
                "reconstructed_points",
                "points_delta",
            ]
            if c in df_position_candidate.columns
        ],
    ].head(100)
)

Mismatches before position correction: 340
Mismatches after high-confidence corrections: 44
Reconciliation after corrections: 99.954247%


,season,player_id,player_name,position,gameweek,minutes,total_points,reconstructed_points,points_delta
1480,2016-17,94,Robert Kenedy Nunes do Nascimento,DEF,37,74,2,1,1
2629,2016-17,169,Jeffrey Schlupp,MID,8,66,1,2,-1
2649,2016-17,169,Jeffrey Schlupp,MID,29,90,7,4,3
2652,2016-17,169,Jeffrey Schlupp,MID,32,90,6,3,3
2653,2016-17,169,Jeffrey Schlupp,MID,33,68,4,5,-1
2657,2016-17,169,Jeffrey Schlupp,MID,36,90,0,2,-2
2658,2016-17,169,Jeffrey Schlupp,MID,37,90,12,9,3
2659,2016-17,169,Jeffrey Schlupp,MID,38,90,1,2,-1
11232,2017-18,123,Jeffrey Schlupp,MID,6,90,-1,1,-2
11233,2017-18,123,Jeffrey Schlupp,MID,7,67,1,2,-1


In [53]:
# ============================================================
# STEP 9G — LOCALIZE REMAINING SCORING MISMATCHES
# ============================================================

remaining_mismatches = df_position_candidate[
    df_position_candidate["points_delta"] != 0
].copy()

remaining_player_seasons = (
    remaining_mismatches
    .groupby(
        [
            "season",
            "player_id",
            "player_name",
            "position",
        ],
        as_index=False,
    )
    .agg(
        mismatch_rows=("points_delta", "size"),
        delta_values=(
            "points_delta",
            lambda s: tuple(sorted(set(s)))
        ),
        min_delta=("points_delta", "min"),
        max_delta=("points_delta", "max"),
        goals=("goals_scored", "sum"),
        clean_sheets=("clean_sheets", "sum"),
    )
    .sort_values(
        "mismatch_rows",
        ascending=False,
    )
)

display(remaining_player_seasons)

print(
    "Remaining mismatching player-seasons:",
    len(remaining_player_seasons),
)

,season,player_id,player_name,position,mismatch_rows,delta_values,min_delta,max_delta,goals,clean_sheets
3,2017-18,222,James Milner,MID,12,"(-2, -1, 3)",-2,3,0,6
2,2017-18,123,Jeffrey Schlupp,MID,11,"(-2, -1, 3)",-2,3,0,3
5,2017-18,510,Declan Rice,MID,10,"(-2, -1, 3)",-2,3,0,3
1,2016-17,169,Jeffrey Schlupp,MID,7,"(-2, -1, 3)",-2,3,0,3
4,2017-18,482,Jairo Riedewald,MID,3,"(-1, 3)",-1,3,0,2
0,2016-17,94,Robert Kenedy Nunes do Nascimento,DEF,1,"(1,)",1,1,0,0


Remaining mismatching player-seasons: 6


In [54]:
# ============================================================
# STEP 9H — FULL PLAYER-SEASON POSITION COMPARISON
# ============================================================

remaining_keys = (
    remaining_player_seasons[
        ["season", "player_id"]
    ]
    .drop_duplicates()
)

remaining_full_rows = (
    df_clean.merge(
        remaining_keys,
        on=["season", "player_id"],
        how="inner",
    )
)

records = []

for (
    season,
    player_id,
    player_name,
    recorded_position,
), group in remaining_full_rows.groupby(
    [
        "season",
        "player_id",
        "player_name",
        "position",
    ]
):

    record = {
        "season": season,
        "player_id": player_id,
        "player_name": player_name,
        "recorded_position": recorded_position,
        "rows": len(group),
    }

    for candidate_position in ["GK", "DEF", "MID", "FWD"]:

        test_group = group.copy()
        test_group["position"] = candidate_position

        reconstructed = test_group.apply(
            reconstruct_fpl_points,
            axis=1,
        )

        matches = (
            reconstructed
            == test_group["total_points"]
        )

        record[
            f"{candidate_position}_matches"
        ] = int(matches.sum())

        record[
            f"{candidate_position}_mismatches"
        ] = int((~matches).sum())

        record[
            f"{candidate_position}_match_pct"
        ] = matches.mean() * 100

    records.append(record)


remaining_position_evidence = (
    pd.DataFrame(records)
)

display(
    remaining_position_evidence.sort_values(
        "rows",
        ascending=False,
    )
)

,season,player_id,player_name,recorded_position,rows,GK_matches,GK_mismatches,GK_match_pct,DEF_matches,DEF_mismatches,DEF_match_pct,MID_matches,MID_mismatches,MID_match_pct,FWD_matches,FWD_mismatches,FWD_match_pct
0,2016-17,94,Robert Kenedy Nunes do Nascimento,DEF,38,37,1,97.368421,37,1,97.368421,38,0,100.000000,38,0,100.000000
1,2016-17,169,Jeffrey Schlupp,MID,38,38,0,100.000000,38,0,100.000000,31,7,81.578947,31,7,81.578947
2,2017-18,123,Jeffrey Schlupp,MID,38,38,0,100.000000,38,0,100.000000,27,11,71.052632,27,11,71.052632
3,2017-18,222,James Milner,MID,38,38,0,100.000000,38,0,100.000000,26,12,68.421053,26,12,68.421053
4,2017-18,482,Jairo Riedewald,MID,38,38,0,100.000000,38,0,100.000000,35,3,92.105263,35,3,92.105263
5,2017-18,510,Declan Rice,MID,38,38,0,100.000000,38,0,100.000000,28,10,73.684211,28,10,73.684211


In [55]:
# ============================================================
# STEP 9I — COMPONENT-LEVEL SCORING BREAKDOWN
# ============================================================

def fpl_point_components(row):
    position = row["position"]
    minutes = row["minutes"]

    appearance = 0

    if minutes > 0:
        appearance += 1

    if minutes >= 60:
        appearance += 1

    goal_points_map = {
        "GK": 6,
        "DEF": 6,
        "MID": 5,
        "FWD": 4,
    }

    goal_points = (
        row["goals_scored"]
        * goal_points_map.get(position, 0)
    )

    assist_points = row["assists"] * 3

    clean_sheet_points = 0

    if minutes >= 60 and row["clean_sheets"]:
        clean_sheet_points = {
            "GK": 4,
            "DEF": 4,
            "MID": 1,
            "FWD": 0,
        }.get(position, 0)

    conceded_points = 0

    if position in {"GK", "DEF"}:
        conceded_points = -int(
            row["goals_conceded"] // 2
        )

    save_points = 0

    if position == "GK":
        save_points = int(
            row["saves"] // 3
        )

    penalty_points = (
        row["penalties_saved"] * 5
        - row["penalties_missed"] * 2
    )

    card_own_goal_points = (
        - row["yellow_cards"]
        - row["red_cards"] * 3
        - row["own_goals"] * 2
    )

    bonus_points = row["bonus"]

    return pd.Series({
        "appearance_pts": appearance,
        "goal_pts": goal_points,
        "assist_pts": assist_points,
        "clean_sheet_pts": clean_sheet_points,
        "conceded_pts": conceded_points,
        "save_pts": save_points,
        "penalty_pts": penalty_points,
        "card_own_goal_pts": card_own_goal_points,
        "bonus_pts": bonus_points,
    })


component_breakdown = (
    remaining_mismatches.apply(
        fpl_point_components,
        axis=1,
    )
)

remaining_diagnostic = pd.concat(
    [
        remaining_mismatches.reset_index(drop=True),
        component_breakdown.reset_index(drop=True),
    ],
    axis=1,
)

display(
    remaining_diagnostic[
        [
            "season",
            "player_id",
            "player_name",
            "position",
            "gameweek",
            "minutes",
            "goals_scored",
            "assists",
            "clean_sheets",
            "goals_conceded",
            "yellow_cards",
            "red_cards",
            "bonus",
            "appearance_pts",
            "goal_pts",
            "assist_pts",
            "clean_sheet_pts",
            "conceded_pts",
            "card_own_goal_pts",
            "bonus_pts",
            "total_points",
            "reconstructed_points",
            "points_delta",
        ]
    ]
)

,season,player_id,player_name,position,gameweek,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,bonus,appearance_pts,goal_pts,assist_pts,clean_sheet_pts,conceded_pts,card_own_goal_pts,bonus_pts,total_points,reconstructed_points,points_delta
0,2016-17,94,Robert Kenedy Nunes do Nascimento,DEF,37,74,0,0,0,3,0,0,0,2,0,0,0,-1,0,0,2,1,1
1,2016-17,169,Jeffrey Schlupp,MID,8,66,0,0,0,2,0,0,0,2,0,0,0,0,0,0,1,2,-1
2,2016-17,169,Jeffrey Schlupp,MID,29,90,0,0,1,0,0,0,1,2,0,0,1,0,0,1,7,4,3
3,2016-17,169,Jeffrey Schlupp,MID,32,90,0,0,1,0,0,0,0,2,0,0,1,0,0,0,6,3,3
4,2016-17,169,Jeffrey Schlupp,MID,33,68,0,1,0,2,0,0,0,2,0,3,0,0,0,0,4,5,-1
5,2016-17,169,Jeffrey Schlupp,MID,36,90,0,0,0,5,0,0,0,2,0,0,0,0,0,0,0,2,-2
6,2016-17,169,Jeffrey Schlupp,MID,37,90,0,1,1,0,0,0,3,2,0,3,1,0,0,3,12,9,3
7,2016-17,169,Jeffrey Schlupp,MID,38,90,0,0,0,2,0,0,0,2,0,0,0,0,0,0,1,2,-1
8,2017-18,123,Jeffrey Schlupp,MID,6,90,0,0,0,5,1,0,0,2,0,0,0,0,-1,0,-1,1,-2
9,2017-18,123,Jeffrey Schlupp,MID,7,67,0,0,0,3,0,0,0,2,0,0,0,0,0,0,1,2,-1


In [56]:
# ============================================================
# STEP 9J — POSITION EVIDENCE FROM DISCRIMINATIVE ROWS
# ============================================================

VALID_POSITIONS = ["GK", "DEF", "MID", "FWD"]

position_votes = []

for _, row in remaining_mismatches.iterrows():

    matching_positions = []

    for candidate in VALID_POSITIONS:
        test_row = row.copy()
        test_row["position"] = candidate

        if reconstruct_fpl_points(test_row) == row["total_points"]:
            matching_positions.append(candidate)

    position_votes.append({
        "season": row["season"],
        "player_id": row["player_id"],
        "player_name": row["player_name"],
        "recorded_position": row["position"],
        "gameweek": row["gameweek"],
        "points_delta": row["points_delta"],
        "matching_positions": tuple(matching_positions),
        "n_matching_positions": len(matching_positions),
    })


position_votes = pd.DataFrame(position_votes)

display(position_votes)

,season,player_id,player_name,recorded_position,gameweek,points_delta,matching_positions,n_matching_positions
0,2016-17,94,Robert Kenedy Nunes do Nascimento,DEF,37,1,"(MID, FWD)",2
1,2016-17,169,Jeffrey Schlupp,MID,8,-1,"(GK, DEF)",2
2,2016-17,169,Jeffrey Schlupp,MID,29,3,"(GK, DEF)",2
3,2016-17,169,Jeffrey Schlupp,MID,32,3,"(GK, DEF)",2
4,2016-17,169,Jeffrey Schlupp,MID,33,-1,"(GK, DEF)",2
5,2016-17,169,Jeffrey Schlupp,MID,36,-2,"(GK, DEF)",2
6,2016-17,169,Jeffrey Schlupp,MID,37,3,"(GK, DEF)",2
7,2016-17,169,Jeffrey Schlupp,MID,38,-1,"(GK, DEF)",2
8,2017-18,123,Jeffrey Schlupp,MID,6,-2,"(GK, DEF)",2
9,2017-18,123,Jeffrey Schlupp,MID,7,-1,"(GK, DEF)",2


In [57]:
# ============================================================
# STEP 9K — AGGREGATE POSITION EVIDENCE BY PLAYER-SEASON
# ============================================================

records = []

for (
    season,
    player_id,
    player_name,
    recorded_position,
), group in position_votes.groupby(
    [
        "season",
        "player_id",
        "player_name",
        "recorded_position",
    ]
):

    record = {
        "season": season,
        "player_id": player_id,
        "player_name": player_name,
        "recorded_position": recorded_position,
        "mismatch_rows": len(group),
    }

    for pos in VALID_POSITIONS:
        record[f"{pos}_support"] = int(
            group["matching_positions"]
            .apply(lambda x: pos in x)
            .sum()
        )

    records.append(record)


remaining_position_support = (
    pd.DataFrame(records)
    .sort_values(
        "mismatch_rows",
        ascending=False,
    )
)

display(remaining_position_support)

,season,player_id,player_name,recorded_position,mismatch_rows,GK_support,DEF_support,MID_support,FWD_support
3,2017-18,222,James Milner,MID,12,12,12,0,0
2,2017-18,123,Jeffrey Schlupp,MID,11,11,11,0,0
5,2017-18,510,Declan Rice,MID,10,10,10,0,0
1,2016-17,169,Jeffrey Schlupp,MID,7,7,7,0,0
4,2017-18,482,Jairo Riedewald,MID,3,3,3,0,0
0,2016-17,94,Robert Kenedy Nunes do Nascimento,DEF,1,0,0,1,1


In [58]:
manual_evidence_corrections = {
    ("2016-17", 94): "MID",   # Robert Kenedy
    ("2016-17", 169): "DEF",  # Jeffrey Schlupp
    ("2017-18", 123): "DEF",  # Jeffrey Schlupp
    ("2017-18", 222): "DEF",  # James Milner
    ("2017-18", 482): "DEF",  # Jairo Riedewald
    ("2017-18", 510): "DEF",  # Declan Rice
}

df_final_position_candidate = df_position_candidate.copy()

for (season, player_id), corrected_position in (
    manual_evidence_corrections.items()
):
    mask = (
        (df_final_position_candidate["season"] == season)
        & (df_final_position_candidate["player_id"] == player_id)
    )

    df_final_position_candidate.loc[
        mask,
        "position",
    ] = corrected_position

In [59]:
df_final_position_candidate["reconstructed_points"] = (
    df_final_position_candidate.apply(
        reconstruct_fpl_points,
        axis=1,
    )
)

df_final_position_candidate["points_delta"] = (
    df_final_position_candidate["total_points"]
    - df_final_position_candidate["reconstructed_points"]
)

remaining = (
    df_final_position_candidate["points_delta"] != 0
)

print(
    "Remaining mismatches:",
    int(remaining.sum()),
)

print(
    "Exact reconciliation rate:",
    f"{(~remaining).mean():.6%}",
)

display(
    df_final_position_candidate.loc[
        remaining,
        [
            "season",
            "player_id",
            "player_name",
            "position",
            "gameweek",
            "total_points",
            "reconstructed_points",
            "points_delta",
        ],
    ]
)

Remaining mismatches: 0
Exact reconciliation rate: 100.000000%


,season,player_id,player_name,position,gameweek,total_points,reconstructed_points,points_delta


In [60]:
# ============================================================
# FINALIZE VALIDATED POSITION CORRECTIONS
# ============================================================

df_clean = df_final_position_candidate.copy()

# Remove temporary scoring audit columns if present
df_clean = df_clean.drop(
    columns=[
        c for c in [
            "reconstructed_points",
            "points_delta",
            "_candidate_position",
        ]
        if c in df_clean.columns
    ]
)

print("Final position corrections applied.")

print(
    "Remaining scoring mismatches:",
    int(
        (
            df_final_position_candidate["points_delta"] != 0
        ).sum()
    )
)

Final position corrections applied.
Remaining scoring mismatches: 0


In [61]:
df_clean["position_recovery_method"] = "observed_normalized"

for (season, player_id), corrected_position in {
    **correction_lookup,
    **manual_evidence_corrections,
}.items():

    mask = (
        (df_clean["season"] == season)
        & (df_clean["player_id"] == player_id)
    )

    df_clean.loc[
        mask,
        "position_recovery_method",
    ] = "scoring_reconciliation"

In [62]:
display(
    df_clean["position_recovery_method"]
    .value_counts()
    .rename_axis("method")
    .reset_index(name="rows")
)

,method,rows
0,observed_normalized,95081
1,scoring_reconciliation,1088


### FPL scoring reconciliation conclusion

The recorded `total_points` field was independently reconstructed from
the available FPL scoring components.

Initial reconciliation:

- 95,829 / 96,169 rows matched exactly.
- 340 mismatches were found.
- All mismatches occurred in 2016-17 and 2017-18.
- Seasons 2020-21, 2021-22 and 2022-23 reconciled at 100%.

The mismatches were traced to historical player-position classification
errors.

Position corrections were evaluated at `(season, player_id)` level,
because FPL positions may legitimately change between seasons.

After applying only position corrections supported by scoring evidence:

- 96,169 / 96,169 rows reconcile exactly.
- Remaining mismatches: 0.
- Exact reconciliation rate: 100%.

The recorded `total_points` values are therefore retained unchanged.
The validated player-season position corrections are accepted as canonical
cleaning transformations.